In [4]:
# Import required libraries
import numpy as np
import pandas as pd
from sklearn.linear_model import PoissonRegressor
from pandemic_model.stats import mcf_pseudo_r2

def check_scale_invariance(X, y, scale_factors=(0.1, 10, 100)):
    """
    Check if Poisson regression predictions are equivalent under scaling of X and y
    
    Parameters
    ----------
    X : array-like
        Input features (mortality rates)
    y : array-like 
        Target variable (GDP loss)
    scale_factors : tuple
        Factors to scale X and y by
        
    Returns
    -------
    dict
        Dictionary containing predictions and differences for each scaling
    """
    results = {}
    
    # Baseline model
    pm_base = PoissonRegressor(alpha=0)
    pm_base.fit(X, y)
    base_preds = pm_base.predict(X)
    results['baseline'] = {
        'predictions': base_preds,
        'r2': mcf_pseudo_r2(y, base_preds)
    }
    
    # Test X scaling
    for sf in scale_factors:
        X_scaled = X + np.log(sf)  # Adding log(sf) since X is already log-transformed
        pm = PoissonRegressor(alpha=0)
        pm.fit(X_scaled, y)
        preds = pm.predict(X_scaled)
        results[f'X_scale_{sf}'] = {
            'predictions': preds,
            'pred_diff': np.abs(preds - base_preds).mean(),
            'r2': mcf_pseudo_r2(y, preds)
        }
    
    # Test y scaling
    for sf in scale_factors:
        y_scaled = y * sf
        pm = PoissonRegressor(alpha=0)
        pm.fit(X, y_scaled)
        preds = pm.predict(X) / sf  # Scale predictions back to original scale
        results[f'y_scale_{sf}'] = {
            'predictions': preds,
            'pred_diff': np.abs(preds - base_preds).mean(),
            'r2': mcf_pseudo_r2(y_scaled, pm.predict(X))
        }
        
    return results

# Load and prepare data
econ_loss_raw = pd.read_excel("../../data/raw/Economic damages source review.xlsx", 
                              sheet_name="Updated numbers")

# Preprocess as in fit_econ_loss_models.py
econ_loss = econ_loss_raw.rename(columns={
    'Fraction GDP losses': 'share_gdp_loss',
    'Mortality (SMU)': 'mortality_smu',
    'Disease': 'disease'
})
econ_loss['pct_gdp_loss'] = econ_loss['share_gdp_loss']
econ_loss = econ_loss[['share_gdp_loss', 'pct_gdp_loss', 'mortality_smu', 'disease']]
econ_loss_clean = econ_loss.dropna(axis=0)

# Prepare X and y
X = np.log(econ_loss_clean[['mortality_smu']])
y = econ_loss_clean['share_gdp_loss']

# Check scale invariance
results = check_scale_invariance(X, y)

# Print results
print("Scale Invariance Test Results:")
print("-" * 50)
print("\nBaseline model R2:", results['baseline']['r2'])

print("\nX scaling tests:")
for k, v in results.items():
    if k.startswith('X_scale'):
        print(f"\n{k}:")
        print(f"Mean absolute difference in predictions: {v['pred_diff']:.8f}")
        print(f"R2: {v['r2']:.4f}")
        
print("\ny scaling tests:")
for k, v in results.items():
    if k.startswith('y_scale'):
        print(f"\n{k}:")
        print(f"Mean absolute difference in predictions: {v['pred_diff']:.8f}")
        print(f"R2: {v['r2']:.4f}")


Scale Invariance Test Results:
--------------------------------------------------

Baseline model R2: 0.12552505731218766

X scaling tests:

X_scale_0.1:
Mean absolute difference in predictions: 0.00000250
R2: 0.1255

X_scale_10:
Mean absolute difference in predictions: 0.00000307
R2: 0.1255

X_scale_100:
Mean absolute difference in predictions: 0.00000444
R2: 0.1255

y scaling tests:

y_scale_0.1:
Mean absolute difference in predictions: 0.00003681
R2: 0.0801

y_scale_10:
Mean absolute difference in predictions: 0.00000465
R2: 0.2412

y_scale_100:
Mean absolute difference in predictions: 0.00000465
R2: 0.4487
